# Tutorial: Exploring coalescent variation across the genome

## 1. Introduction

**1.1 Background**

With the increasing ease to generate large-scale genomic datasets, our ability to retrace evolutionary history based on DNA sequences has never been more profound. Traditional phylogenetic studies relied on one or a few molecular markers, but it is now becoming commonplace to survey phylogenetic patterns across 100s to 1000s of genetic loci. Nearing a genome-wide view on evolutionary history has major consequences for the field of phylogenetics since phylogenetic history often varies across the genome. In other words, a phylogenetic tree in one region of the genome may vary substantially in topology from a phylogenetic tree based on genetic data from a different region of the genome. A major objective in evolutionary and phylogenetic research is to understand the processes that lead to such discordance and to distinguish between neutral and non-neutral evolutionary dynamics.

***Empirical example demonstrating how the frequency of competing phylogenies can vary across chromosomes and between chromosomes (e.g. sex vs. autosomes)***
![example.png](https://github.com/MozesBlom/tutorials/tree/main/2023_Phy_Eco_Evol/img/example.png)

Coalescent theory presents a population genetic framework that can describe the expected differences in inheritance patterns between genomic loci based on neutral processes alone. Coalescent models aim to describe the probability that any two individuals in a population(s) share a common ancestor at a given time in the past and can easily be extended to multiple populations/species. Explicit demographic parameters, which we will explore in detail, change the probability of a coalescent event (i.e. two individuals share a common ancestor in the previous generation). Coalescent models can therefore be leveraged to generate null-models under neutral evolution and can be compared to empirical observations (i.e. phylogenomic data). In this computer practical, we use a coalescent based simulator to simulate tree sequences across a genomic region and demonstrate how the expected variation in coalescent histories change depending on several demographic properties.

**1.2 Practical overview**

This computer practical is styled in a Jupyter Notebook (NB) format (see Github [README](https://github.com/MozesBlom/tutorials/tree/main/2023_Phy_Eco_Evol) for more details). In short, Jupyter NBs contain either text cells (such as the current cell) in Markdown format or code cells (frequently) in Python3 format. The aim of this practical is not to provide an exhaustive introduction into Python/Markdown/Jupyter, but to provide an introduction into phylogenomic inference, how to account for genome wide variation in inheritance histories (coalescent variation) and what may cause such variation. Therefore, all code to run this practical is already in place and we will only make minor changes to the code itself to demonstrate the phylogenetic consequences of changing specific model parameters.

**1.3 Requirements**

To run this practical, the following Python modules are needed:
* Python 3+
    * numpy 1.10+
    * [msprime](https://msprime.readthedocs.io/en/stable/)
    * matplotlib

## 2. Getting Started

**2.1 Importing modules**

The Python modules mentioned under *1.3 Requirements* first need to be imported before proceeding with the rest of the NB. If running on a Google Colab instance (see [Github](https://github.com/MozesBlom/tutorials/tree/main/2023_Phy_Eco_Evol) for further details) modules will always need to be installed. If running on your personal desktop, in JupyterLab for example, then this may not be needed if a conda environment is already loaded and [selected](https://github.com/jupyterlab/jupyterlab-desktop/blob/master/user-guide.md) with the correct modules installed. NOTE, if you go the latter route then make sure that JupyterLab itself and all dependencies are installed in the relevant Conda environment. Otherwise, the environment will not pop up in the JupyterLab Desktop environment list and cannot be selected. In all other user cases, first proceed with downloading the required Python modules:

In [ ]:
# Download the necessary Python modules. If already present in Conda environment, then skip this step
%pip install msprime 

In [ ]:
# Import the modules
import msprime
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import SVG

## 3. Simulating ancestry with msprime

msprime is a backward-in-time coalescent simulator that can be used to simulate trees and sequences under explicit demographic scenarios. It computes the probability that any two individuals, their haplotypes to be precise, share a common ancestor at a given time in the past. In a single population, under neutral circumstances, this is primarily determined by effective population size. Between populations, the probability of coalescence is further determined by the rate of migration between populations and the time of divergence. While used in a population context, the same principles are relevant when thinking about phylogenetic history between species and is therefore a suitable exercise to explore the determinants of coalescent variation across genomes.

### 3.1 The Tree Sequence class

In [ ]:
# Let's run a simple simulation using msprime
ts = msprime.sim_ancestry(samples=2, random_seed=1)
SVG(ts.draw_svg())

**QUESTION**: We specified to run a simulation with 2 individuals, why does the resulting phylogeny/tree include four tips?

**ANSWER**: ..

In [ ]:
# Let's first dive deeper into the tree sequence that msprime generated
ts

msprime didn't simply generate an image of a phylogeny, but instead produced a summary of the demographic simulation which in this case describes a single phylogeny and all associated meta-information. For example, the number of individuals (`Individuals`), branches (`edges`), migration events (`Migrations`) etc. Importantly note that in this specific model, we simulated a tree sequence for a single site (`Sequence Length` = 1) only. `Sequence Length` can be interpreted as the number of continuous basepairs on a single contig or chromosome. Let's run a simulation for a longer sequence, say 10000 base pairs (bp).

In [ ]:
ts = msprime.sim_ancestry(
                samples=2,
                random_seed=1,
                sequence_length=10000)
SVG(ts.draw_svg())

In [ ]:
ts

Besides the `Sequence Length` nothing really changed. In example, we still only see one tree and the topology is exactly the same. This is because we haven't simulated any recombination events yet. **Without recombination, the evolutionary history of a genomic stretch will always be identical and we therefore do not expect any different coalescent history along a genomic sequence**! Recombination can be included in our simulation by specifying a `recombination_rate`: The probability of a recombination event per genomic unit (bp) per generation.

In [ ]:
ts = msprime.sim_ancestry(
                samples=2,
                random_seed=1,
                sequence_length=10000,
                recombination_rate = 1e-4)
SVG(ts.draw_svg())

This is already more biologically meaningful: Recombination has introduced breakpoints along a chromosome and the coalescent history on each side of the breakpoint is different. Note, as you can see that does not always mean that the topology (order of diversification) changes. In some instances only the tree height, the time to the most recent common ancestor differs (which may be difficult to see in some trees). Now let's have another look at the `Tree Sequence` python class:

In [ ]:
ts

By invoking recombination, the number of `Trees`, `Edges`, `Nodes` all have changed but all this information is still captured in a single `Tree Sequence` class. The `Tree Sequence` class is a helpful Python recipe to greatly reduce the computational footprint when studying phylogenies. In example, consider that we only simulated a 10000 bp. sequence here, but many genomes tend to be billions of bp. in size and studies can now include 1000s of individuals! The `Tree Sequence` class doesn't store every single phylogeny at any given site in a chromosome, but instead notes the (recombination) break points along a sequence where trees differ and only records the branches in the phylogeny that differ between break points and the differences in node heights.

However, besides the computational advantages of using the `Tree Sequence` class, there's also an important biological interpretation here as well. Evolutionary history along a chromosome, coalescent history, only varies between sites if a **recombination** event has occurred. Otherwise adjacent sites in a given sequence will share the same evolutionary history. Most phylogenetic methods are designed to reconstruct a single evolutionary history for a given sequence alignment and thus assume that this segment is free from recombination.

**EXERCISE**: Explore the role of recombination for modeling coalescent variation by changing the frequency of recombination in the code below. What happens when you increase the frequency and what happens if you decrease the frequency?

**ANSWER**: ...

In [ ]:
ts = msprime.sim_ancestry(
                samples=2,
                random_seed=1,
                sequence_length=10000,
                recombination_rate = 1e-4)

print("The total number of trees simulated by changing the recombination frequency: " + str(ts.num_trees))

### 3.2 Effective population size

In the first section, we simulated the coalescent history for two individuals in a single population of a constant size. However, as mentioned before, population size is one of the major determinants that changes the probability of coalescence. Consider the following:

Assume an isolated island population of constant size (100 individuals) with random mating: What is the probability that two individuals share a common ancestor in the previous generation? What if it is a much bigger island with a larger constant population size over time (n = 1000)?

Let's explore how differences in (effective) population size influence our coalescent history using msprime

In [ ]:
# Let's run two simulations and only vary Ne using a new parameter
ts1 = msprime.sim_ancestry(
                samples=2,
                random_seed=1,
                sequence_length=10000,
                population_size=10,
                recombination_rate = 1e-5)

ts2 = msprime.sim_ancestry(
                samples=2,
                random_seed=1,
                sequence_length=10000,
                population_size=100,
                recombination_rate = 1e-5)

In [ ]:
SVG(ts1.draw_svg())

In [ ]:
SVG(ts2.draw_svg())

Comparing the trees between both simulations isn't straightforward because the scale on the y-axis isn't the same. When invoking the `population_size` argument the branch lengths are automatically scaled to units of generations. We can then compare the differences in coalescent times by looking at the first tree for example.

In [ ]:
tree1 = ts1.first()
print("Total branch length:", tree1.total_branch_length)
print("Time at root:", ts1.tables.nodes.time[tree1.root])

In [ ]:
tree2 = ts2.first()
print("Total branch length:", tree2.total_branch_length)
print("Time at root:", ts2.tables.nodes.time[tree2.root])

**QUESTION**: What has happened? Under what circumstances do you expect a longer wait time until the Most Recent Common Ancestor of all individuals?

**ANSWER**: ...

### 3.3 A multi-population/species model

Up till now, we have only modeled coalescent variation across a chromosome within a single population/species. We have seen that phylogenies can vary in both divergence times and topology, under neutral circumstances. However, when employing phylogenetic approaches, we are often interested in describing the relationship between species. We can establish similar models using msprime and use this to demonstrate how gene trees (a phylogeny for a given gene or genomic region) can differ from the underlying species tree.

To run a multi-population model in msprime, we first need to describe the demographic model. Let's simulate the following:

- 4 species
- 2 individuals for each species
- constant population size over time
- no migration/gene flow between species
- equal divergence time between major splits

In [ ]:
# To do so in msprime, we need to describe a demography object which then can be passed to `sim_ancestry`
Ne = 100
dem = msprime.Demography()
dem.add_population(name="human", initial_size=Ne)
dem.add_population(name="chimp", initial_size=Ne)
dem.add_population(name="gorilla", initial_size=Ne)
dem.add_population(name="mouse", initial_size=Ne)

# Table with basic demography model of extant species
dem

What we have created is a demographic model of the extant species. Running a simulation on this would run forever, since the simulation only stops when ALL individuals across all species have coalesced. In this model, this would never be the case because there would be no coalescent events between species. So let's specify the divergence times and ancestral populations:

In [ ]:
# Add ancestral populations and the divergence times when they split off
div_t = 1000
dem.add_population(name="AncestralPopulation1", initial_size=Ne)
dem.add_population_split(time=div_t, derived=["human","chimp"], ancestral="AncestralPopulation1")
dem.add_population(name="AncestralPopulation2", initial_size=Ne)
dem.add_population_split(time=div_t*2, derived=["AncestralPopulation1","gorilla"], ancestral="AncestralPopulation2")
dem.add_population(name="AncestralPopulation3", initial_size=Ne)
dem.add_population_split(time=div_t*3, derived=["AncestralPopulation2","mouse"], ancestral="AncestralPopulation3")
dem

Note that the ancestral populations have now been included and we added three events: Each corresponding to a divergence time.

We can now create a simulation for this specific demographic model. Let's first simulate for a single tree

In [ ]:
sample_size = 2

ts = msprime.sim_ancestry(
                samples={"human" : sample_size, "chimp" : sample_size, "gorilla" : sample_size, "mouse" : sample_size, "AncestralPopulation1" : 0, "AncestralPopulation2" : 0, "AncestralPopulation3" : 0},
                demography=dem,
                random_seed=1)
display(SVG(ts.first().draw(width=500, height=400)))

Since we are now working with a larger number of species and individuals, the phylogeny has expanded but it's not informative with regards to which tip belongs to which species. However, since we are working with species, there is now a new population/species column which can be used to colour the nodes by species

In [ ]:
colour_map = {0:"red", 1:"blue", 2:"green", 3:"orange", 4:"purple", 5:"black", 6:"yellow", 7:"pink", 8:"brown", 9:"gray"}
node_colours = {u.id: colour_map[u.population] for u in ts.nodes()}
# The code below will only work in a Jupyter notebook with SVG output enabled.
display(SVG(ts.first().draw(node_colours=node_colours, width=500, height=400)))

Under the present model specifications, we have simulated a single phylogeny where we sampled two individuals per species and reconstructed the coalescent history of a single locus. In this simulation, all haplotypes belonging to the same species (coloured nodes here) share a MRCA and the gene tree will be identical to the species tree. However, we have already seen that effective population size has a strong effect on the probability that two haplotypes share a common ancestor in the previous generation and here we will demonstrate that this can lead to mismatches between gene and species tree.

**EXERCISE**: Increase the effective population size in the following model and simulate ancestry. Try a few values, what happens?

**ANSWER**: ...

In [ ]:
# Model parameters
Ne = 10000
div_t = 1000
sample_size = 2

# ------------------ #

dem = msprime.Demography()
dem.add_population(name="human", initial_size=Ne)
dem.add_population(name="chimp", initial_size=Ne)
dem.add_population(name="gorilla", initial_size=Ne)
dem.add_population(name="mouse", initial_size=Ne)

dem.add_population(name="AncestralPopulation1", initial_size=Ne)
dem.add_population_split(time=div_t, derived=["human","chimp"], ancestral="AncestralPopulation1")
dem.add_population(name="AncestralPopulation2", initial_size=Ne)
dem.add_population_split(time=div_t*2, derived=["AncestralPopulation1","gorilla"], ancestral="AncestralPopulation2")
dem.add_population(name="AncestralPopulation3", initial_size=Ne)
dem.add_population_split(time=div_t*3, derived=["AncestralPopulation2","mouse"], ancestral="AncestralPopulation3")

ts = msprime.sim_ancestry(
                samples={"human" : sample_size, "chimp" : sample_size, "gorilla" : sample_size, "mouse" : sample_size, "AncestralPopulation1" : 0, "AncestralPopulation2" : 0, "AncestralPopulation3" : 0},
                demography=dem,
                random_seed=1)

colour_map = {0:"red", 1:"blue", 2:"green", 3:"orange", 4:"purple", 5:"black", 6:"yellow", 7:"pink", 8:"brown", 9:"gray"}
node_colours = {u.id: colour_map[u.population] for u in ts.nodes()}
# The code below will only work in a Jupyter notebook with SVG output enabled.
display(SVG(ts.first().draw(node_colours=node_colours, width=500, height=400)))

The process you have simulated above is commonly known as **deep coalescence** or **incomplete lineage sorting** (ILS). In such cases, gene trees will not follow the species tree and a coalescence event between two haplotypes of the same species predates a species split. In the above example, you have only changed the effective population size but the probability of ILS also depends on other parameters. **Question**: How could we change the model above, without changing the effective population size parameter, so that all haplotypes belonging to the same species share a MRCA?

In [ ]:
# Model parameters
Ne = 10000
div_t = 1000
sample_size = 2

# ------------------ #

dem = msprime.Demography()
dem.add_population(name="human", initial_size=Ne)
dem.add_population(name="chimp", initial_size=Ne)
dem.add_population(name="gorilla", initial_size=Ne)
dem.add_population(name="mouse", initial_size=Ne)

dem.add_population(name="AncestralPopulation1", initial_size=Ne)
dem.add_population_split(time=div_t, derived=["human","chimp"], ancestral="AncestralPopulation1")
dem.add_population(name="AncestralPopulation2", initial_size=Ne)
dem.add_population_split(time=div_t*2, derived=["AncestralPopulation1","gorilla"], ancestral="AncestralPopulation2")
dem.add_population(name="AncestralPopulation3", initial_size=Ne)
dem.add_population_split(time=div_t*3, derived=["AncestralPopulation2","mouse"], ancestral="AncestralPopulation3")

ts = msprime.sim_ancestry(
                samples={"human" : sample_size, "chimp" : sample_size, "gorilla" : sample_size, "mouse" : sample_size, "AncestralPopulation1" : 0, "AncestralPopulation2" : 0, "AncestralPopulation3" : 0},
                demography=dem,
                random_seed=1)

colour_map = {0:"red", 1:"blue", 2:"green", 3:"orange", 4:"purple", 5:"black", 6:"yellow", 7:"pink", 8:"brown", 9:"gray"}
node_colours = {u.id: colour_map[u.population] for u in ts.nodes()}
# The code below will only work in a Jupyter notebook with SVG output enabled.
display(SVG(ts.first().draw(node_colours=node_colours, width=500, height=400)))

Up till now we have simulated a single phylogeny only because we excluded the recombination parameter. Let's include recombination and investigate how coalescence histories can vary across a chromosome. To simplify the simulation time, we reduce the effective population size and divergence time. 

In [ ]:
# Model parameters
Ne = 10
div_t = 10
sample_size = 1

# ------------------ #

dem = msprime.Demography()
dem.add_population(name="human", initial_size=Ne)
dem.add_population(name="chimp", initial_size=Ne)
dem.add_population(name="gorilla", initial_size=Ne)
dem.add_population(name="mouse", initial_size=Ne)

dem.add_population(name="AncestralPopulation1", initial_size=Ne)
dem.add_population_split(time=div_t, derived=["human","chimp"], ancestral="AncestralPopulation1")
dem.add_population(name="AncestralPopulation2", initial_size=Ne)
dem.add_population_split(time=div_t*2, derived=["AncestralPopulation1","gorilla"], ancestral="AncestralPopulation2")
dem.add_population(name="AncestralPopulation3", initial_size=Ne)
dem.add_population_split(time=div_t*3, derived=["AncestralPopulation2","mouse"], ancestral="AncestralPopulation3")

ts = msprime.sim_ancestry(
                samples={"human" : sample_size, "chimp" : sample_size, "gorilla" : sample_size, "mouse" : sample_size, "AncestralPopulation1" : 0, "AncestralPopulation2" : 0, "AncestralPopulation3" : 0},
                demography=dem,
                sequence_length=10000,
                recombination_rate = 1e-4,
                random_seed=1)

colour_map = {0:"red", 1:"blue", 2:"green", 3:"orange", 4:"purple", 5:"black", 6:"yellow", 7:"pink", 8:"brown", 9:"gray"}
node_colours = {u.id: colour_map[u.population] for u in ts.nodes()}
# The code below will only work in a Jupyter notebook with SVG output enabled.
SVG(ts.draw_svg())

In [ ]:
# If you want to look at the first tree only, coloured by species association:
display(SVG(ts.first().draw(node_colours=node_colours, width=500, height=400)))

**SUMMARY**: With the above simulations, we have demonstrated that effective population size and divergence time are important determinants that predict the probability of incomplete lineage sorting and that this can vary across a chromosome since recombination leads to distinct units that can have a different evolutionary history: Even under neutral circumstances (i.e. without selection, introgression, etc.). Modern day phylogenomic studies need to account for this variation in coalescent history and determine what processes have led to this discordance. For example, have a look at the distribution of topologies in the empirical example introduced above:

<img src="img/example.png" width="500" height="500">

When comparing autosomes with the sex chromosomes, there is much stronger support for a single topology on the sex chromosome (i.e. less ILS): The Z-chromosome in birds. Could you come up with a neutral explanation why that may be? Hint: Birds are diploid organisms and the Z-chromosome is equivalent to the X-chromosome in humans.